# 9.8 Quantized inference (GPTQ, AWQ)

Quantized inference trades tiny numerical error for much smaller, faster language models.

Quantized inference sits on Transformer machinery and asks how low-bit weights change memory, logits, and task behavior. We use tiny CPU-only arrays to reproduce the lesson arithmetic, then stress the same quantizer across a D1-D5 ladder with rare salient channels.

Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

SEED = 908
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)


def build_quantization_ladder():
    rng = np.random.default_rng(SEED)
    ladders = []
    weights_d1 = np.array([0.37, -0.12, 0.20, 2.00])
    X_d1 = np.array([[1.0, 0.4, 0.2, 1.0]])
    y_d1 = (X_d1 @ weights_d1 > 1.0).astype(int)
    ladders.append({"name": "D1 one prompt", "X": X_d1, "y": y_d1, "w": weights_d1, "salient": [3]})

    X_d2 = rng.normal(size=(16, 6))
    weights_d2 = np.array([0.7, -0.4, 0.3, 0.2, -0.6, 0.5])
    y_d2 = (X_d2 @ weights_d2 > 0.0).astype(int)
    ladders.append({"name": "D2 few-shot classifier", "X": X_d2, "y": y_d2, "w": weights_d2, "salient": [0, 5]})

    X_d3 = rng.normal(size=(32, 8))
    X_d3[:4, 6] += 5.0
    weights_d3 = np.array([0.35, -0.25, 0.18, 0.12, -0.20, 0.28, 2.40, -0.15])
    y_d3 = (X_d3 @ weights_d3 > np.median(X_d3 @ weights_d3)).astype(int)
    ladders.append({"name": "D3 rare large channel", "X": X_d3, "y": y_d3, "w": weights_d3, "salient": [6]})

    texts = [
        "short helpful answer",
        "long detailed policy answer",
        "pricing question with rare currency",
        "creative image prompt",
        "budget pacing alert",
        "plain greeting",
        "rare legal qualifier",
        "conversion summary"
    ]
    vocab = ["short", "long", "policy", "pricing", "rare", "creative", "budget", "legal", "conversion"]
    X_d4 = np.zeros((len(texts), len(vocab)))
    for i, text in enumerate(texts):
        for j, token in enumerate(vocab):
            X_d4[i, j] = float(token in text)
    weights_d4 = np.array([0.3, -0.2, 0.6, 0.4, 1.8, 0.2, 0.5, 2.1, 0.4])
    y_d4 = (X_d4 @ weights_d4 > 0.9).astype(int)
    ladders.append({"name": "D4 inline text features", "X": X_d4, "y": y_d4, "w": weights_d4, "salient": [4, 7]})

    n = 80
    d = 12
    X_d5 = rng.normal(scale=0.7, size=(n, d))
    X_d5[:, 10] = 0.0
    X_d5[::10, 10] = 7.0
    weights_d5 = rng.normal(scale=0.18, size=d)
    weights_d5[10] = 2.8
    weights_d5[3] = -1.9
    logits_d5 = X_d5 @ weights_d5
    y_d5 = (logits_d5 > np.quantile(logits_d5, 0.55)).astype(int)
    ladders.append({"name": "D5 long context scaling", "X": X_d5, "y": y_d5, "w": weights_d5, "salient": [3, 10]})
    return ladders


def accuracy_from_logits(logits, labels):
    predictions = (logits > 0.0).astype(int)
    return float(np.mean(predictions == labels))


## The concept, built once (D1)

The lesson formula is $$\hat w=s(q-z),\quad q=\mathrm{clip}(\mathrm{round}(w/s)+z,0,2^b-1).$$
For the worked weight $w=0.37$, scale $s=0.1$, zero point $z=8$, and $b=4$, the integer level should be $q=12$, the dequantized value should be $\hat w=0.4$, and the absolute error should be $0.03$.

In [ ]:

def quantize_dequantize(weights, bits=4, scale=0.1, zero_point=8, protected=None):
    protected = set(protected or [])
    q_min = 0
    q_max = (2 ** bits) - 1
    q = np.clip(np.round(weights / scale) + zero_point, q_min, q_max).astype(int)
    dequantized = scale * (q - zero_point)
    protected_mask = np.zeros_like(weights, dtype=bool)
    for index in protected:
        protected_mask[index] = True
    dequantized = np.where(protected_mask, weights, dequantized)
    return q, dequantized, protected_mask

q_demo, w_hat_demo, protected_demo = quantize_dequantize(np.array([0.37]), bits=4, scale=0.1, zero_point=8)
levels = 2 ** 4
error_demo = abs(0.37 - float(w_hat_demo[0]))
fp16_gb = 7_000_000_000 * 16 / 8 / 1_000_000_000
int4_gb = 7_000_000_000 * 4 / 8 / 1_000_000_000
assert levels == 16
assert int(q_demo[0]) == 12
assert round(float(w_hat_demo[0]), 1) == 0.4
assert round(error_demo, 2) == 0.03
assert round(fp16_gb, 1) == 14.0
assert round(int4_gb, 1) == 3.5
print(levels, q_demo[0], w_hat_demo[0], error_demo, fp16_gb, int4_gb)


AWQ-style protection keeps selected salient channels in higher precision while ordinary channels are quantized. In the lesson, the large weight $2.0$ is protected while the smaller weight $0.2$ is quantized.

In [ ]:

weights_awq = np.array([2.0, 0.2])
q_awq, w_awq, protected_awq = quantize_dequantize(weights_awq, bits=4, scale=0.1, zero_point=8, protected=[0])
assert protected_awq.tolist() == [True, False]
assert float(w_awq[0]) == 2.0
assert round(float(w_awq[1]), 1) == 0.2
print("q", q_awq)
print("dequantized", w_awq)


## The dataset ladder

The F8 ladder grows from one prompt-like vector to a longer context simulation with rare high-activation channels. Each rung keeps labels from the full-precision model so the same quantized inference method can be evaluated without downloads.

In [ ]:

ladder = build_quantization_ladder()
for rung in ladder:
    print(rung["name"], "X", rung["X"].shape, "classes", np.bincount(rung["y"]), "salient", rung["salient"])
    print("sample", rung["X"][0])


## Run the same quantizer across D1-D5

We report compression ratio as the primary deployment metric and include accuracy/error diagnostics so compression is never separated from task behavior.

In [ ]:

results = []
for rung in ladder:
    q, w_quantized, mask = quantize_dequantize(rung["w"], bits=4, scale=0.1, zero_point=8)
    logits_full = rung["X"] @ rung["w"]
    logits_quantized = rung["X"] @ w_quantized
    accuracy = accuracy_from_logits(logits_quantized, rung["y"])
    mean_error = float(np.mean(np.abs(logits_full - logits_quantized)))
    compression_ratio = 16 / 4
    results.append({"name": rung["name"], "metric": compression_ratio, "accuracy": accuracy, "error": mean_error, "weights": rung["w"], "quantized": w_quantized})

print("rung | compression_ratio | accuracy | mean_logit_error")
for result in results:
    print(f"{result['name']} | {result['metric']:.1f}x | {result['accuracy']:.3f} | {result['error']:.3f}")


## Results visualization

The left panels show weight error per rung; the right panel shows the compression curve from D1 to D5.

In [ ]:

fig, axes = plt.subplots(1, len(results), figsize=(16, 3))
for ax, result in zip(axes, results):
    error = result["quantized"] - result["weights"]
    ax.bar(np.arange(len(error)), error)
    ax.axhline(0.0, color="black", linewidth=0.8)
    ax.set_title(result["name"].split()[0])
    ax.set_xlabel("channel")
    ax.set_ylabel("quant error")
plt.tight_layout()

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(range(1, len(results) + 1), [item["metric"] for item in results], marker="o")
ax.set_xticks(range(1, len(results) + 1))
ax.set_xticklabels(["D1", "D2", "D3", "D4", "D5"])
ax.set_ylabel("compression ratio")
ax.set_title("4-bit vs fp16 storage")
plt.show()


## Pitfall on the hardest rung

Pitfall: quantizing all weights equally damages rare large activation channels. Reproduce that on D5, then protect salient channels AWQ-style.

In [ ]:

d5 = ladder[-1]
q_plain, w_plain, mask_plain = quantize_dequantize(d5["w"], bits=4, scale=0.1, zero_point=8)
q_awq, w_awq, mask_awq = quantize_dequantize(d5["w"], bits=4, scale=0.1, zero_point=8, protected=d5["salient"])
full_logits = d5["X"] @ d5["w"]
plain_logits = d5["X"] @ w_plain
awq_logits = d5["X"] @ w_awq
plain_error = float(np.mean(np.abs(full_logits - plain_logits)))
awq_error = float(np.mean(np.abs(full_logits - awq_logits)))
plain_accuracy = accuracy_from_logits(plain_logits, d5["y"])
awq_accuracy = accuracy_from_logits(awq_logits, d5["y"])
print("plain", plain_error, plain_accuracy)
print("awq", awq_error, awq_accuracy)


## Evaluate it + Practice

- Metric: compression ratio, with accuracy and mean logit error as guardrails.
- No-skill baseline: fp16 inference has 1x compression and no quantization error.
- Cheap sanity check: the worked $0.37$ weight must map to integer 12 and dequantize to 0.4.
- Ablation: turn off AWQ protection on D5 and rare-channel error should rise.
- Failure signals: large per-channel errors, accuracy drops, or scale/zero-point choices missing from the report.

Practice prompts:


**Practice.** Change the scale from 0.1 to 0.05 and predict which D5 channels clip.

**Practice.** Protect a different channel and compare the D5 logit-error printout.

**Practice.** Replace accuracy with perplexity-like negative log likelihood for the same logits.